# 0. Pyroton V2 Step 3 pilot

Select a free Colab T4 runtime. Run the cells in order, authorize Drive in cell 1,
and upload the Step 3 inputs ZIP in cell 2. A completed Step 2b report is required.
This run trains only on its audited, length-matched ORPO pairs and evaluates only
dev and MBPP+. Outputs and resumable journals live under `My Drive/Pyroton/V2-step3-pilot`.
The 32-draw probe covers the original 167 zero-pass problems and does not feed training.


In [ ]:
# 1. Verify a T4 and connect persistent Drive storage
from pathlib import Path
from google.colab import drive, files
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
from urllib.request import urlopen
from zipfile import ZIP_DEFLATED, ZipFile

gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True)
if "T4" not in gpu:
    raise RuntimeError("Change the runtime to T4 GPU before continuing.")
mount = Path("/content/drive")
my_drive = mount / "MyDrive"
if not my_drive.is_dir():
    drive.mount(str(mount))
if not my_drive.is_dir():
    raise RuntimeError("Authorize Drive in Colab and rerun cell 1.")
run = my_drive / "Pyroton" / "V2-step3-pilot"
run.mkdir(parents=True, exist_ok=True)
local = Path("/content/pyroton-step3-pilot")
local.mkdir(exist_ok=True)
print(gpu.strip(), run)


In [ ]:
# 2. Upload and verify the sealed Step 3 inputs
name = "pyroton-v2-step3-pilot-inputs.zip"
bundle = run / name
if not bundle.exists():
    upload = files.upload()
    if name not in upload:
        raise ValueError(f"Select {name}.")
    temporary = bundle.with_suffix(".tmp")
    temporary.write_bytes(upload[name])
    temporary.replace(bundle)

def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

with ZipFile(bundle) as archive:
    manifest = json.loads(archive.read("manifest.json"))
    if set(archive.namelist()) != set(manifest) | {"manifest.json"} or archive.testzip() is not None:
        raise ValueError("The bundle has incomplete or invalid members.")
    for member, digest in manifest.items():
        path = (local / member).resolve()
        if not path.is_relative_to(local.resolve()) or hashlib.sha256(archive.read(member)).hexdigest() != digest:
            raise ValueError(f"Invalid bundle member {member}.")
    archive.extractall(local)

training = local / "data"
bench = local / "benchmarks"
scripts = local / "scripts"
eval_inputs = local / "heldout"
assert {p.name for p in eval_inputs.iterdir()} == {"dev.json", "mbppplus.json"}
pairs, quality, cpu_noise = (training / name for name in (
    "orpo-length-matched.jsonl", "quality-report.json", "stock-noise.json"))
eval_dir, probe_dir, exported = (run / name for name in ("evaluation", "probe", "models"))
for directory in (eval_dir, probe_dir, exported):
    directory.mkdir(exist_ok=True)

def python(script, *args):
    subprocess.run([sys.executable, str(script), *map(str, args)], cwd=local, check=True)

checkpoint_dir = run / "checkpoints"
train_args = ["--pairs", pairs, "--quality-report", quality, "--noise-report", cpu_noise,
              "--output-dir", checkpoint_dir]
python(scripts / "train_step3_pilot.py", *train_args, "--validate-only")
print(json.loads(quality.read_bytes())["length_matched_pairs"])


In [ ]:
# 3. Install the pinned training and scoring libraries
from importlib.metadata import PackageNotFoundError, version

pins = ["trl==0.29.0", "transformers==4.57.6", "accelerate==1.12.0", "peft==0.18.1",
        "datasets==4.5.0", "huggingface-hub==0.36.0", "tokenizers==0.22.2", "fsspec==2025.10.0",
        "wasi-python==0.1.5", "wasmtime==50.0.1", "sentencepiece==0.2.1", "xxhash==3.6.0",
        "multiprocess==0.70.18", "dill==0.4.0", "safetensors==0.7.0", "pyarrow==22.0.0"]
runtime_pins = {}
for library in ("torch", "torchao"):
    try:
        runtime_pins[library] = version(library)
    except PackageNotFoundError:
        pass
constraints = local / "runtime-constraints.txt"
constraints.write_text("\n".join(f"{name}=={value}" for name, value in runtime_pins.items()) + "\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", str(constraints), *pins], check=True)
if any(version(name) != value for name, value in runtime_pins.items()):
    raise RuntimeError("A protected Colab runtime package changed version.")
subprocess.run([sys.executable, "-c", "import datasets, peft, torch, wasi_python; "
                "from trl.experimental.orpo import ORPOConfig, ORPOTrainer; "
                "print(torch.__version__, torch.cuda.get_device_name(0))"], check=True)
os.environ["PYROTON_WASI_CACHE"] = str(local / "wasi-cache")


In [ ]:
# 4. Download the pinned stock GGUF and llama.cpp backend
def fetch(url, path, expected):
    if path.is_file() and sha(path) == expected:
        return path
    temporary = path.with_name(path.name + ".download")
    with urlopen(url, timeout=120) as response, temporary.open("wb") as output:
        shutil.copyfileobj(response, output, length=4 * 1024 * 1024)
    if sha(temporary) != expected:
        raise RuntimeError(f"SHA-256 mismatch for {path.name}.")
    temporary.replace(path)
    return path

stock_sha = "cc324af070c2ecbfd324a30884d2f951a7ff756aba85cb811a6ec436933bb046"
stock_url = ("https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF/resolve/"
             "f86cb2c1fa58255f8052cc32aeede1b7482d4361/qwen2.5-coder-1.5b-instruct-q4_k_m.gguf")
cache = run / "runtime-cache"
cache.mkdir(exist_ok=True)
stock = local / "stock-Q4_K_M.gguf"
stock_cache = fetch(stock_url, cache / stock.name, stock_sha)
if not stock.exists() or sha(stock) != stock_sha:
    shutil.copyfile(stock_cache, stock)
assets = {
    "llama-b12521-bin-ubuntu-cuda-12.8-x64.tar.gz": "02509065d54065aeff9f71fd15ef22c2f1804742f6823e75cfd67d9e6cdff84a",
    "cudart-llama-b12521-bin-ubuntu-cuda-12.8-x64.tar.gz": "ce5352857ebf9119786f73af2cdcd7fca3d0f0cd826bfeee0567bfe23ebdc81a",
}
backend = local / "backend"
backend.mkdir(exist_ok=True)
for asset, hash_value in assets.items():
    url = "https://github.com/ggml-org/llama.cpp/releases/download/b12521/" + asset
    with tarfile.open(fetch(url, cache / asset, hash_value), "r:gz") as archive:
        archive.extractall(backend, filter="data")
server, = backend.rglob("llama-server")
quantize, = backend.rglob("llama-quantize")
os.environ["LD_LIBRARY_PATH"] = ":".join(sorted({str(p.parent) for p in backend.rglob("*.so*")})
                                         + [os.environ.get("LD_LIBRARY_PATH", "")])
subprocess.run([str(server), "--version"], check=True)
converter_repo = local / "llama.cpp"
if not converter_repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "b12521",
                    "https://github.com/ggml-org/llama.cpp.git", str(converter_repo)], check=True)
converter = converter_repo / "convert_hf_to_gguf.py"


In [ ]:
# 5. Measure both stock repeats on the T4 before training
suites = {"dev": eval_inputs / "dev.json", "mbppplus": eval_inputs / "mbppplus.json"}

def evaluate(model, prefix, dataset):
    generations = prefix.with_suffix(".jsonl")
    scores = prefix.with_name(prefix.name + "-scores.json")
    python(bench / "generate_holdout.py", "--model", model, "--server", server,
           "--dataset", dataset, "--output", generations, "--tokens", 512,
           "--slots", 1, "--threads", 2, "--gpu-layers", 99)
    python(bench / "score_holdout.py", "--dataset", dataset, "--generations", generations, "--output", scores)

def eval_report(destination, *extra):
    python(bench / "step3_eval_report.py", "--dev", suites["dev"], "--mbppplus", suites["mbppplus"],
           "--run-dir", eval_dir, "--output", destination, *extra)

for repeat in (1, 2):
    for suite, dataset in suites.items():
        evaluate(stock, eval_dir / f"stock-{suite}-{repeat}", dataset)
gpu_noise = eval_dir / "stock-t4-noise.json"
eval_report(gpu_noise)


In [ ]:
# 6. Run or resume the two-epoch ORPO pilot
if not gpu_noise.is_file():
    raise RuntimeError("Complete the matched stock T4 comparisons in cell 5 first.")
python(scripts / "train_step3_pilot.py", *train_args)


In [ ]:
# 7. Merge and quantize both epoch checkpoints
python(scripts / "export_step3_gguf.py", "--checkpoint-root", checkpoint_dir,
       "--converter", converter, "--quantize", quantize, "--scratch", local / "conversion",
       "--output-dir", exported)


In [ ]:
# 8. Evaluate and compare both Q4_K_M checkpoints
for epoch in (1, 2):
    checkpoint = exported / f"epoch-{epoch}-Q4_K_M.gguf"
    model = local / checkpoint.name
    stamp = json.loads(checkpoint.with_suffix(".json").read_bytes())
    if not model.exists() or sha(model) != stamp["gguf_sha256"]:
        shutil.copyfile(checkpoint, model)
    if sha(model) != stamp["gguf_sha256"]:
        raise RuntimeError(f"Failed to verify epoch {epoch} model copy.")
    for suite, dataset in suites.items():
        evaluate(model, eval_dir / f"epoch-{epoch}-{suite}", dataset)
training_report = eval_dir / "step3-checkpoint-report.json"
eval_report(training_report, "--checkpoints", "epoch-1", "epoch-2")


In [ ]:
# 9. Sample the original 167 zero-pass problems on T4
cohort = training / "zero-pass-cohort.json"
cohort_tasks = probe_dir / "verified-zero-pass.json"
zero_script = bench / "step3_zero_pass.py"
python(zero_script, "--cohort", cohort, "--corrected-corpus", training / "corrected-corpus.json",
       "--output", cohort_tasks)
samples = probe_dir / "stock-32-draws.jsonl"
judgments = probe_dir / "stock-32-judgments.jsonl"
python(bench / "sample_pilot.py", "--model", stock, "--server", server, "--verified-corpus", cohort_tasks,
       "--output", samples, "--draws-per-style", 16, "--gpu-layers", 99, "--slots", 12, "--threads", 2)
python(bench / "judge_pilot.py", "--verified-corpus", cohort_tasks, "--samples", samples,
       "--output", judgments, "--draws-per-style", 16)
probe_report = probe_dir / "zero-pass-probe-report.json"
python(zero_script, "--cohort", cohort, "--corrected-corpus", cohort_tasks,
       "--judgments", judgments, "--stock-model-sha256", stock_sha, "--output", probe_report)


In [ ]:
# 10. Package the reports and stop
report_zip = run / "pyroton-v2-step3-pilot-reports.zip"
with ZipFile(report_zip, "w", ZIP_DEFLATED) as archive:
    for path in (quality, cpu_noise, gpu_noise, training_report, probe_report,
                 *(exported / f"epoch-{epoch}-Q4_K_M.json" for epoch in (1, 2))):
        archive.write(path, path.name)
print(report_zip)
print("The two models and complete epoch checkpoints are saved on Drive.")
files.download(str(report_zip))
